# Chronos-2 Backend Quantization (MASE ONNXRuntime)

This notebook quantizes a robust Chronos2-derived proxy subgraph using MASE's ONNX quantizer (`Quantizer`) and benchmarks FP32 vs INT8 ONNXRuntime.


In [ ]:
import time
from collections import Counter
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
import os

os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))


from chop.models import get_model
from chop.passes.graph.transforms.onnxrt.quantize import Quantizer

OUTPUT_DIR = Path("artifacts/backend_quant")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"
MODEL_NAME = "chronos-2"
BATCH_SIZE = 1
WARMUP = 10
ITERS = 100


In [ ]:
# Load Chronos2
model = get_model(MODEL_NAME, pretrained=True).eval().to(DEVICE)

cfg = model.chronos_config
INPUT_PATCH_SIZE = int(cfg.input_patch_size)
INPUT_PATCH_STRIDE = int(cfg.input_patch_stride)
OUTPUT_PATCH_SIZE = int(cfg.output_patch_size)
NUM_QUANTILES = int(model.num_quantiles)
CONTEXT_LEN = int(cfg.context_length)

if INPUT_PATCH_STRIDE != INPUT_PATCH_SIZE:
    print(
        f"[WARN] input_patch_stride ({INPUT_PATCH_STRIDE}) != input_patch_size ({INPUT_PATCH_SIZE}). "
        "Proxy uses non-overlapping patch view for robustness."
    )

print("Loaded model:", MODEL_NAME)
print(
    {
        "context_length": CONTEXT_LEN,
        "input_patch_size": INPUT_PATCH_SIZE,
        "input_patch_stride": INPUT_PATCH_STRIDE,
        "output_patch_size": OUTPUT_PATCH_SIZE,
        "num_quantiles": NUM_QUANTILES,
        "device": DEVICE,
    }
)


Loaded model: chronos-2
{'context_length': 8192, 'input_patch_size': 16, 'input_patch_stride': 16, 'output_patch_size': 16, 'num_quantiles': 21, 'device': 'cpu'}


In [ ]:
# Chronos2 derived backend proxy
class Chronos2BackendProxy(torch.nn.Module):
    def __init__(self, core_model: torch.nn.Module, context_len: int):
        super().__init__()
        self.core_model = core_model
        self.input_patch_embedding = core_model.input_patch_embedding
        self.output_patch_embedding = core_model.output_patch_embedding

        self.input_patch_size = int(core_model.chronos_config.input_patch_size)
        self.output_patch_size = int(core_model.chronos_config.output_patch_size)
        self.num_quantiles = int(core_model.num_quantiles)

        self.context_len = int(context_len)
        self.trim_len = (self.context_len // self.input_patch_size) * self.input_patch_size
        self.num_patches = self.trim_len // self.input_patch_size

        te = torch.linspace(-1.0, 0.0, steps=self.trim_len, dtype=torch.float32)
        te = te.view(1, self.num_patches, self.input_patch_size)
        self.register_buffer("time_enc", te, persistent=False)

    def forward(self, context: torch.Tensor) -> torch.Tensor:
        x = context.to(torch.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected context shape [B, T], got {tuple(x.shape)}")
        if x.shape[1] < self.trim_len:
            raise ValueError(f"Context length {x.shape[1]} is smaller than required trim_len {self.trim_len}")

        x = x[:, -self.trim_len :]
        bsz = x.shape[0]

        patches = x.view(bsz, self.num_patches, self.input_patch_size)
        patch_mask = torch.ones_like(patches)
        time_enc = self.time_enc.expand(bsz, -1, -1)

        # Matches Chronos input_patch_embedding input contract: [time, values, mask]
        emb_in = torch.cat([time_enc, patches, patch_mask], dim=-1)
        patch_emb = self.input_patch_embedding(emb_in)

        # aggregate patch sequence before output head
        pooled = patch_emb.mean(dim=1, keepdim=True)
        head_out = self.output_patch_embedding(pooled)  # [B, 1, num_quantiles * output_patch_size]
        out = head_out.view(bsz, self.num_quantiles, self.output_patch_size)
        return out


In [ ]:
# Prepare dummy input and sanity forward
proxy = Chronos2BackendProxy(model, context_len=CONTEXT_LEN).eval().to(DEVICE)
dummy_context = torch.randn(BATCH_SIZE, CONTEXT_LEN, device=DEVICE, dtype=torch.float32)

with torch.inference_mode():
    sanity_out = proxy(dummy_context)

print("Dummy context shape:", tuple(dummy_context.shape))
print("Proxy output shape:", tuple(sanity_out.shape))


Dummy context shape: (1, 8192)
Proxy output shape: (1, 21, 16)


In [ ]:
# ONNX export helper

def export_onnx_proxy(proxy_model: torch.nn.Module, dummy_input: torch.Tensor, fp32_path: Path) -> str:
    fp32_path.parent.mkdir(parents=True, exist_ok=True)

    common_kwargs = dict(
        input_names=["context"],
        output_names=["quantile_preds"],
    )

    # try classic exporter
    try:
        torch.onnx.export(
            proxy_model,
            (dummy_input,),
            str(fp32_path),
            opset_version=17,
            do_constant_folding=True,
            **common_kwargs,
        )
        onnx_model = onnx.load(str(fp32_path))
        onnx.checker.check_model(onnx_model)
        _ = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
        print("ONNX export succeeded with classic exporter (opset 17).")
        return "classic"
    except Exception as classic_ex:
        print(f"Classic exporter failed ({type(classic_ex).__name__}): {classic_ex}")
        print("Retrying dynamo exporter (opset 18)...")

    # Fallback to modern exporter
    torch.onnx.export(
        proxy_model,
        (dummy_input,),
        str(fp32_path),
        opset_version=18,
        dynamo=True,
        **common_kwargs,
    )

    onnx_model = onnx.load(str(fp32_path))
    onnx.checker.check_model(onnx_model)
    _ = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    print("ONNX export succeeded with dynamo exporter (opset 18).")
    return "dynamo"



In [ ]:
# MASE ONNX dynamic INT8 quantization helper

def quantize_onnx_with_mase_quantizer(fp32_path: Path, int8_path: Path, prep_path: Path) -> None:
    config = {
        "accelerator": "cpu",
        "default": {"config": {"precision": "int8"}},
    }

    q = Quantizer(config)

    # ORT pre-process can fail on some exported graphs (observed NoneType/HasField issue).
    # Fall back to direct dynamic quantization, which is still valid and often robust.
    quant_input_path = fp32_path
    try:
        q.pre_process(fp32_path, prep_path)
        quant_input_path = prep_path
        print(f"Pre-processed ONNX written to: {prep_path}")
    except Exception as exc:
        print(f"[WARN] pre_process failed ({type(exc).__name__}): {exc}")
        print("[WARN] Falling back to direct dynamic quantization on FP32 ONNX.")

    q.quantize_dynamic(quant_input_path, int8_path)

    _ = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])



In [ ]:
# Benchmark helper

def benchmark_ort(onnx_path: Path, ort_inputs: dict, warmup: int, iters: int, providers=None):
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    session = ort.InferenceSession(
        str(onnx_path),
        sess_options=sess_opts,
        providers=providers or ["CPUExecutionProvider"],
    )

    for _ in range(warmup):
        _ = session.run(None, ort_inputs)

    t0 = time.perf_counter()
    last_out = None
    for _ in range(iters):
        last_out = session.run(None, ort_inputs)
    t1 = time.perf_counter()

    avg_ms = (t1 - t0) * 1000.0 / iters
    return avg_ms, last_out[0]


In [ ]:
# Quantization audit helpers

def collect_onnx_quantization_stats(onnx_path: Path):
    m = onnx.load(str(onnx_path))
    op_counts = Counter(node.op_type for node in m.graph.node)

    quant_ops = {
        "MatMulInteger",
        "QLinearMatMul",
        "QuantizeLinear",
        "DequantizeLinear",
        "DynamicQuantizeLinear",
    }
    quant_op_count = sum(op_counts.get(op, 0) for op in quant_ops)

    dtype_counts = Counter(
        onnx.TensorProto.DataType.Name(init.data_type) for init in m.graph.initializer
    )

    return {
        "op_counts": op_counts,
        "quant_op_count": quant_op_count,
        "dtype_counts": dtype_counts,
    }


def total_onnx_artifact_size(onnx_path: Path) -> int:
    total = 0
    if onnx_path.exists():
        total += onnx_path.stat().st_size
    sidecar = Path(str(onnx_path) + ".data")
    if sidecar.exists():
        total += sidecar.stat().st_size
    return total


In [ ]:
fp32_path = OUTPUT_DIR / "proxy_fp32.onnx"
prep_path = OUTPUT_DIR / "proxy_fp32_preprocessed.onnx"
int8_path = OUTPUT_DIR / "proxy_int8_dynamic.onnx"

export_mode = export_onnx_proxy(proxy, dummy_context, fp32_path)
quantize_onnx_with_mase_quantizer(fp32_path, int8_path, prep_path)

ort_inputs = {"context": dummy_context.detach().cpu().numpy().astype(np.float32)}
providers = ["CPUExecutionProvider"]

fp32_ms, fp32_out = benchmark_ort(fp32_path, ort_inputs, warmup=WARMUP, iters=ITERS, providers=providers)
int8_ms, int8_out = benchmark_ort(int8_path, ort_inputs, warmup=WARMUP, iters=ITERS, providers=providers)

mae = float(np.mean(np.abs(fp32_out - int8_out)))
max_abs_diff = float(np.max(np.abs(fp32_out - int8_out)))
speedup = fp32_ms / int8_ms if int8_ms > 0 else float("inf")

fp32_stats = collect_onnx_quantization_stats(fp32_path)
int8_stats = collect_onnx_quantization_stats(int8_path)

fp32_size = total_onnx_artifact_size(fp32_path)
int8_size = total_onnx_artifact_size(int8_path)

print("=== Backend Quantization Summary ===")
print(f"Export mode: {export_mode}")
print(f"FP32 latency: {fp32_ms:.3f} ms")
print(f"INT8 latency: {int8_ms:.3f} ms")
print(f"Speedup: {speedup:.3f}x")
print(f"MAE: {mae:.8f}")
print(f"Max abs diff: {max_abs_diff:.8f}")
print(f"FP32 total artifact size: {fp32_size / (1024 * 1024):.2f} MB")
print(f"INT8 total artifact size: {int8_size / (1024 * 1024):.2f} MB")
print(f"INT8 quantized op count: {int8_stats['quant_op_count']}")

if int8_stats["quant_op_count"] == 0:
    print("[WARN] No quantization ops were inserted. Quantization appears non-effective for this exported graph.")



C:\Users\Maxim\AppData\Local\Temp\ipykernel_28992\2044851608.py:26: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[1] < self.trim_len:


ONNX export succeeded with classic exporter (opset 17).
Pre-processed ONNX written to: artifacts\backend_quant\proxy_fp32_preprocessed.onnx
=== Backend Quantization Summary ===
Export mode: classic
FP32 latency: 6.594 ms
INT8 latency: 2.666 ms
Speedup: 2.473x
MAE: 0.00298430
Max abs diff: 0.01278235
FP32 total artifact size: 23.72 MB
INT8 total artifact size: 6.02 MB
INT8 quantized op count: 12


In [ ]:
# diagnostics

def top_ops(counter_obj: Counter, k: int = 20):
    return counter_obj.most_common(k)

print("=== Top FP32 ops ===")
for name, count in top_ops(fp32_stats["op_counts"], k=20):
    print(f"{name:>24}: {count}")

print("=== Top INT8 ops ===")
for name, count in top_ops(int8_stats["op_counts"], k=20):
    print(f"{name:>24}: {count}")

print("=== Initializer dtype counts (INT8 model) ===")
for dtype, count in int8_stats["dtype_counts"].most_common():
    print(f"{dtype:>16}: {count}")

if int8_stats["quant_op_count"] == 0:
    print("Interpretation: backend quantization did not rewrite this graph effectively.")
elif speedup < 1.0:
    print("Interpretation: quantized graph exists, but runtime overhead/kernel assignment made it slower on this setup.")
else:
    print("Interpretation: quantization appears effective on this setup.")

=== Top FP32 ops ===
                Constant: 11
                     Add: 8
                  MatMul: 6
                 Reshape: 2
                    Relu: 2
                    Cast: 1
                   Slice: 1
         ConstantOfShape: 1
                     Mul: 1
                   Equal: 1
                   Where: 1
                  Expand: 1
                  Concat: 1
              ReduceMean: 1
=== Top INT8 ops ===
                 Reshape: 14
                     Mul: 12
                     Add: 8
   DynamicQuantizeLinear: 6
           MatMulInteger: 6
                    Cast: 6
                    Relu: 2
                   Slice: 1
                  Concat: 1
              ReduceMean: 1
=== Initializer dtype counts (INT8 model) ===
           INT64: 17
           FLOAT: 14
            INT8: 12
Interpretation: quantization appears effective on this setup.
